In [21]:
# Ensure required packages are installed into the notebook kernel's Python

import sys
import subprocess

pkgs = ['pdfplumber', 'pandas', 'requests']

print('Installing packages into', sys.executable)

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    *pkgs
])

print('Packages installed (or already present)')

Installing packages into C:\Users\nknav\AppData\Local\Programs\Python\Python311\python.exe
Packages installed (or already present)


In [31]:
import pdfplumber
import pandas as pd
import re
import requests
from pathlib import Path
print('imports ok')

imports ok


In [27]:
# Configuration: set your PDF path and password (if any)
PDF_PATH = Path('APR2026_AA09940389_TXN.pdf')  # change as needed
PASSWORD = 'KTYPK4362N' #if your PDF is password-protected
assert PDF_PATH.exists(), f'PDF not found: {PDF_PATH}'
print('PDF found:', PDF_PATH)

PDF found: APR2026_AA09940389_TXN.pdf


In [28]:
#Reads mutual fund statement and converts it to table in csv

# ---------------------------------------------------
# CONFIG
# ---------------------------------------------------


# ---------------------------------------------------
# EXTRACT TEXT FROM PDF
# ---------------------------------------------------

all_text = ""

with pdfplumber.open(PDF_PATH, password=PASSWORD) as pdf:

    print("Total pages:", len(pdf.pages))

    for i, page in enumerate(pdf.pages):

        try:

            text = page.extract_text(layout=True)

            if text:
                print(f"Page {i+1}: extracted {len(text)} chars")

                all_text += "\n"
                all_text += text

            else:
                print(f"Page {i+1}: NO TEXT")

        except Exception as e:
            print(f"Page {i+1}: ERROR")
            print(e)

# ---------------------------------------------------
# CLEAN TEXT
# ---------------------------------------------------

text = all_text.replace("\xa0", " ")
with open("full_extracted_text.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

lines = [line.strip() for line in text.split("\n") if line.strip()]

# ---------------------------------------------------
# PARSE TABLE
# ---------------------------------------------------

rows = []

scheme_buffer = []
pending_row = None        # row waiting for potential trailing name lines
post_isin_buffer = []     # trailing scheme-name lines collected after the ISIN row

NEW_SCHEME_PATTERN = r'^[A-Z0-9]{2,10}\s*-\s*'

SKIP_KEYWORDS = [
    "Scheme Name",
    "Closing",
    "Profit/Loss",
    "Page ",
    "MUTUAL FUND UNITS",
    "Depository",
    "ISIN Folio",
    "Central Depository",
    "Unrealised",
    "Valuation",
]


def flush_pending(pending_row, post_isin_buffer, rows):
    """
    Commit a pending row to the result list, appending any trailing
    scheme-name continuation lines that were collected after the ISIN row.
    """
    if pending_row is None:
        return
    trailing = " ".join(post_isin_buffer).strip()
    if trailing:
        pending_row["Scheme Name"] = re.sub(
            r'\s+', ' ',
            pending_row["Scheme Name"] + " " + trailing
        ).strip()
    rows.append(pending_row)


for line in lines:

    # ------------------------------------------------
    # Skip headers / junk lines
    # A skip line also acts as a "flush" boundary –
    # any pending row is committed before we move on.
    # ------------------------------------------------

    if any(x in line for x in SKIP_KEYWORDS):
        flush_pending(pending_row, post_isin_buffer, rows)
        pending_row = None
        post_isin_buffer = []
        continue

    # ------------------------------------------------
    # Detect ISIN row
    # ------------------------------------------------

    isin_match = re.search(r'(INF[A-Z0-9]{9})', line)

    if isin_match:

        # Flush any previously pending row before starting a new one
        flush_pending(pending_row, post_isin_buffer, rows)
        pending_row = None
        post_isin_buffer = []

        try:

            isin = isin_match.group(1)

            # Exact ISIN position
            isin_start = line.find(isin)

            before_isin = line[:isin_start]
            after_isin  = line[isin_start + len(isin):]

            # Build scheme name from buffered pre-ISIN lines + text before ISIN
            scheme_name = " ".join(scheme_buffer + [before_isin])

            # Fix merged word boundary just before ISIN, e.g. "GrowthINF846..."
            scheme_name = re.sub(
                r'([a-z])([A-Z]{2,})$',
                r'\1 \2',
                scheme_name
            )

            # Normalise whitespace
            scheme_name = re.sub(r'\s+', ' ', scheme_name).strip()

            # ------------------------------------------------
            # Parse remaining values
            # ------------------------------------------------

            values = after_isin.strip().split()

            # Expected format after ISIN:
            # folio units nav invested valuation pnl pnl%

            if len(values) < 7:
                scheme_buffer = []
                continue

            # Parse from RIGHT side
            pnl_pct    = values[-1]
            pnl        = values[-2]
            valuation  = values[-3]
            invested   = values[-4]
            nav        = values[-5]
            units      = values[-6]

            # Everything before units is folio
            folio = " ".join(values[:-6])

            # ------------------------------------------------
            # Store row as *pending* – do NOT append yet.
            # Trailing scheme-name lines may still follow.
            # ------------------------------------------------

            pending_row = {
                "Scheme Name":         scheme_name,
                "ISIN":                isin,
                "Folio":               folio,
                "Closing Bal (Units)": float(units.replace(",", "")),
                "NAV":                 float(nav.replace(",", "")),
                "Invested Amount":     float(invested.replace(",", "")),
                "Valuation":           float(valuation.replace(",", "")),
                "PnL":                 float(pnl.replace(",", "")),
                "PnL %":               float(pnl_pct.replace(",", ""))
            }

            # Reset pre-ISIN buffer
            scheme_buffer = []

        except Exception as e:

            print("\nFAILED TO PARSE:")
            print(line)
            print(e)

            scheme_buffer = []

    else:

        if pending_row is not None:

            # ------------------------------------------------
            # POST-ISIN mode:
            # Lines here are either trailing scheme-name words
            # OR the start of a new scheme.
            # ------------------------------------------------

            if re.match(NEW_SCHEME_PATTERN, line):

                # New scheme begins – flush the pending row first
                flush_pending(pending_row, post_isin_buffer, rows)
                pending_row     = None
                post_isin_buffer = []

                # Start fresh buffer for the new scheme
                scheme_buffer = [line]

            else:
                # Continuation of the current scheme's name
                post_isin_buffer.append(line)

        else:

            # ------------------------------------------------
            # PRE-ISIN mode: build up scheme name buffer
            # ------------------------------------------------

            if re.match(NEW_SCHEME_PATTERN, line):
                # Start fresh buffer
                scheme_buffer = [line]

            else:
                # Continue multi-line scheme name
                scheme_buffer.append(line)


# Flush whatever is still pending at end of input
flush_pending(pending_row, post_isin_buffer, rows)

# ---------------------------------------------------
# CREATE DATAFRAME
# ---------------------------------------------------

df = pd.DataFrame(rows)

# ---------------------------------------------------
# DISPLAY
# ---------------------------------------------------

print("\n========== OUTPUT ==========\n")

print(df.head(20))

print("\nTotal rows extracted:", len(df))

# ---------------------------------------------------
# SAVE CSV
# ---------------------------------------------------

df.to_csv("mutual_funds.csv", index=False)

print("\nSaved to mutual_funds.csv")

Total pages: 26
Page 1: extracted 7277 chars
Page 2: extracted 9819 chars
Page 3: extracted 5429 chars
Page 4: extracted 5670 chars
Page 5: extracted 5647 chars
Page 6: extracted 5646 chars
Page 7: extracted 5658 chars
Page 8: extracted 5639 chars
Page 9: extracted 5648 chars
Page 10: extracted 5639 chars
Page 11: extracted 5639 chars
Page 12: extracted 5641 chars
Page 13: extracted 5641 chars
Page 14: extracted 11154 chars
Page 15: extracted 10735 chars
Page 16: extracted 9927 chars
Page 17: extracted 7760 chars
Page 18: extracted 21828 chars
Page 19: extracted 14785 chars
Page 20: extracted 14494 chars
Page 21: extracted 14459 chars
Page 22: extracted 16871 chars
Page 23: extracted 11325 chars
Page 24: extracted 11090 chars
Page 25: extracted 8620 chars
Page 26: extracted 11429 chars

FAILED TO PARSE:
INF179KC1HO1                    3965.662  --    --     -- 3965.662 18.815 74,613.93
could not convert string to float: '--'

FAILED TO PARSE:
INF457M01133                     150.000  -

In [29]:
import pandas as pd

# ---------------------------------------------------
# READ CSV
# ---------------------------------------------------

df = pd.read_csv("mutual_funds.csv")

# ---------------------------------------------------
# CREATE Scheme_Name_Short
# ---------------------------------------------------

# Example:
# EVG1 - Groww Nifty EV & New Age Automotive ETF FOF - Direct Plan - Growth
#
# We want:
# Groww Nifty EV & New Age Automotive ETF FOF

df['Scheme_Name_Short'] = (
    df['Scheme Name']
    .str.split("-")
    .str[1]
    .str.strip()
)

# ---------------------------------------------------
# CREATE Key1 and Key2
# ---------------------------------------------------

split_cols = df['Scheme_Name_Short'].str.split(" ", n=1, expand=True)

df['Key1'] = split_cols[0]

df['Key2'] = split_cols[1]

# ---------------------------------------------------
# SAVE
# ---------------------------------------------------

df.to_csv("mutual_funds_with_key.csv", index=False)

print(df.head())

                                         Scheme Name          ISIN  \
0   OGDG - Axis Large & Mid Cap Fund - Direct Growth  INF846K01J46   
1        ICDG - BARODA BNP PARIBAS INDIA CONSUMPTION  INF251K01PT7   
2  896 - DSP Multi Asset Allocation Fund - Direct...  INF740KA1TE9   
3  896 - DSP Multi Asset Allocation Fund - Direct...  INF740KA1TE9   
4  EVG1 - Groww Nifty EV & New Age Automotive ETF...  INF666M01IL4   

            Folio  Closing Bal (Units)      NAV  Invested Amount  Valuation  \
0  910176197453/0              186.280  36.3800           5000.0    6776.87   
1    3105845992/0              181.910  33.1023           5000.0    6021.64   
2     10948456/93             4971.773  16.4019          80000.0   81546.52   
3      8637956/70             1999.900  16.4019          20000.0   32802.16   
4   51011465425/0              999.950   9.2011          10000.0    9200.64   

        PnL  PnL %                            Scheme_Name_Short    Key1  \
0   1776.87  35.54           

In [32]:
#It reads all schemes from MFAPI site and saves them in table in csv

# ---------------------------------------------------
# DOWNLOAD DATA FROM MFAPI
# ---------------------------------------------------

url = "https://api.mfapi.in/mf"

print("Downloading data from MFAPI...")

response = requests.get(url)

response.raise_for_status()

data = response.json()

print(f"Total records received: {len(data)}")

# ---------------------------------------------------
# CONVERT TO DATAFRAME
# ---------------------------------------------------

df = pd.DataFrame(data)

# ---------------------------------------------------
# CREATE SHORT NAME
# ---------------------------------------------------

# Example:
# "Axis Small Cap Fund - Direct Plan - Growth"

df['Scheme_Name_Short'] = (
    df['schemeName']
    .str.split("-")
    .str[0]
    .str.strip()
)

# ---------------------------------------------------
# CREATE key1 and key2
# ---------------------------------------------------

# key1 = first word/company
# key2 = remaining scheme text
def get_priority(name):

    name = str(name).lower()

    has_direct = "direct" in name
    has_growth = "growth" in name

    if has_direct and has_growth:
        return 2

    elif has_direct or has_growth:
        return 1

    else:
        return 0


split_cols = df['Scheme_Name_Short'].str.split(" ", n=1, expand=True)

df['Key1'] = split_cols[0]

df['Key2'] = split_cols[1]
df['Priority'] = df['schemeName'].apply(get_priority)

# ---------------------------------------------------
# DISPLAY
# ---------------------------------------------------

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

# ---------------------------------------------------
# SAVE
# ---------------------------------------------------

df.to_csv("mfapi_all_schemes.csv", index=False)

print("\nSaved to: mfapi_all_schemes.csv")

Total records received: 37605

Columns:
['schemeCode', 'schemeName', 'isinGrowth', 'isinDivReinvestment', 'Scheme_Name_Short', 'Key1', 'Key2', 'Priority']

First 5 Rows:
   schemeCode                                         schemeName isinGrowth  \
0      100027  Grindlays Super Saver Income Fund-GSSIF-Half Y...       None   
1      100028  Grindlays Super Saver Income Fund-GSSIF-Quater...       None   
2      100029     Grindlays Super Saver Income Fund-GSSIF-Growth       None   
3      100030  Grindlays Super Saver Income Fund-GSSIF-Annual...       None   
4      100031  Grindlays Super Saver Income Fund-GSSIF - ST-D...       None   

  isinDivReinvestment                  Scheme_Name_Short       Key1  \
0                None  Grindlays Super Saver Income Fund  Grindlays   
1                None  Grindlays Super Saver Income Fund  Grindlays   
2                None  Grindlays Super Saver Income Fund  Grindlays   
3                None  Grindlays Super Saver Income Fund  Grindlays   


In [33]:
my_mf=pd.read_csv("mutual_funds_with_key.csv")
mf_api=pd.read_csv("mfapi_all_schemes.csv")

print(my_mf.head(2))

print(mf_api.head(2))

                                        Scheme Name          ISIN  \
0  OGDG - Axis Large & Mid Cap Fund - Direct Growth  INF846K01J46   
1       ICDG - BARODA BNP PARIBAS INDIA CONSUMPTION  INF251K01PT7   

            Folio  Closing Bal (Units)      NAV  Invested Amount  Valuation  \
0  910176197453/0               186.28  36.3800           5000.0    6776.87   
1    3105845992/0               181.91  33.1023           5000.0    6021.64   

       PnL  PnL %                     Scheme_Name_Short    Key1  \
0  1776.87  35.54             Axis Large & Mid Cap Fund    Axis   
1  1021.64  20.43  BARODA BNP PARIBAS INDIA CONSUMPTION  BARODA   

                            Key2  
0           Large & Mid Cap Fund  
1  BNP PARIBAS INDIA CONSUMPTION  
   schemeCode                                         schemeName isinGrowth  \
0      100027  Grindlays Super Saver Income Fund-GSSIF-Half Y...        NaN   
1      100028  Grindlays Super Saver Income Fund-GSSIF-Quater...        NaN   

  isinDiv

In [34]:
import pandas as pd

# ---------------------------------------------------
# CLEAN KEYS
# ---------------------------------------------------

my_mf['Key1'] = my_mf['Key1'].astype(str).str.lower().str.strip()
my_mf['Key2'] = my_mf['Key2'].astype(str).str.lower().str.strip()

mf_api['Key1'] = mf_api['Key1'].astype(str).str.lower().str.strip()
mf_api['Key2'] = mf_api['Key2'].astype(str).str.lower().str.strip()

# ---------------------------------------------------
# FUNCTION FOR PARTIAL MATCH
# ---------------------------------------------------

def is_partial_match(my_key2, api_key2):

    return (
        my_key2 in api_key2
        or api_key2 in my_key2
    )

# ---------------------------------------------------
# MATCHING
# ---------------------------------------------------

matched_rows = []

for _, my_row in my_mf.iterrows():

    my_key1 = my_row['Key1']
    my_key2 = my_row['Key2']

    # -----------------------------------------------
    # 100% match on Key1
    # -----------------------------------------------

    candidates = mf_api[
        mf_api['Key1'] == my_key1
    ].copy()

    # -----------------------------------------------
    # Partial match on Key2
    # -----------------------------------------------

    candidates = candidates[
        candidates['Key2'].apply(
            lambda x: is_partial_match(my_key2, x)
        )
    ]

    # -----------------------------------------------
    # Keep highest Priority
    # -----------------------------------------------

    if not candidates.empty:

        candidates = candidates.sort_values(
            by='Priority',
            ascending=False
        )

        best_match = candidates.iloc[0]

        merged_row = {
            **my_row.to_dict(),
            **best_match.to_dict()
        }

    else:

        merged_row = my_row.to_dict()

    matched_rows.append(merged_row)

# ---------------------------------------------------
# FINAL DATAFRAME
# ---------------------------------------------------

final_df = pd.DataFrame(matched_rows)

print(final_df.head())

# ---------------------------------------------------
# SAVE
# ---------------------------------------------------

final_df.to_csv("matched_mutual_funds.csv", index=False)

print("\nSaved to matched_mutual_funds.csv")

                                         Scheme Name          ISIN  \
0   OGDG - Axis Large & Mid Cap Fund - Direct Growth  INF846K01J46   
1        ICDG - BARODA BNP PARIBAS INDIA CONSUMPTION  INF251K01PT7   
2  896 - DSP Multi Asset Allocation Fund - Direct...  INF740KA1TE9   
3  896 - DSP Multi Asset Allocation Fund - Direct...  INF740KA1TE9   
4  EVG1 - Groww Nifty EV & New Age Automotive ETF...  INF666M01IL4   

            Folio  Closing Bal (Units)      NAV  Invested Amount  Valuation  \
0  910176197453/0              186.280  36.3800           5000.0    6776.87   
1    3105845992/0              181.910  33.1023           5000.0    6021.64   
2     10948456/93             4971.773  16.4019          80000.0   81546.52   
3      8637956/70             1999.900  16.4019          20000.0   32802.16   
4   51011465425/0              999.950   9.2011          10000.0    9200.64   

        PnL  PnL %                            Scheme_Name_Short    Key1  \
0   1776.87  35.54           

In [41]:
import pandas as pd
import requests
import time

# ---------------------------------------------------
# READ YOUR FILE
# ---------------------------------------------------

df = pd.read_csv("matched_mutual_funds.csv")

print(df.columns.tolist())

# Expected columns:
# Scheme Code
# Closing Bal (Units)

# ---------------------------------------------------
# FUNCTION TO FETCH LATEST NAV
# ---------------------------------------------------

def get_latest_nav(scheme_code):

    try:

        url = f"https://api.mfapi.in/mf/{scheme_code}"

        response = requests.get(url, timeout=20)

        response.raise_for_status()

        data = response.json()

        meta = data.get("meta", {})
        nav_data = data.get("data", [])

        if not nav_data:
            return None

        latest = nav_data[0]

        return {
            "Latest_Date": latest.get("date"),
            "Latest_NAV": float(latest.get("nav"))
        }

    except Exception as e:

        print(f"Failed for scheme code: {scheme_code}")
        print(e)

        return None

# ---------------------------------------------------
# FETCH NAVS
# ---------------------------------------------------

latest_values = []

for idx, row in df.iterrows():

    scheme_code = row.get("schemeCode")

    units = row.get("Closing Bal (Units)", 0)

    print(f"\nProcessing Scheme Code: {scheme_code}")

    if pd.isna(scheme_code):
        latest_values.append({})
        continue

    nav_info = get_latest_nav(int(scheme_code))

    if nav_info:

        latest_nav = nav_info["Latest_NAV"]

        current_value = units * latest_nav

        latest_values.append({
            "Latest_Date": nav_info["Latest_Date"],
            "Latest_NAV": latest_nav,
            "Current_Value": round(current_value, 2)
        })

    else:

        latest_values.append({})

    time.sleep(0.2)

# ---------------------------------------------------
# MERGE RESULTS
# ---------------------------------------------------

latest_df = pd.DataFrame(latest_values)

final_df = pd.concat([df, latest_df], axis=1)

# ---------------------------------------------------
# APPROX BUY NAV
# ---------------------------------------------------

final_df['Approx_Buy_NAV'] = (
    final_df['Invested Amount']
    / final_df['Closing Bal (Units)']
)

# ---------------------------------------------------
# CURRENT PNL
# ---------------------------------------------------

final_df['Current_PnL'] = (
    (final_df['Latest_NAV'] - final_df['Approx_Buy_NAV'])
    * final_df['Closing Bal (Units)']
)

# ---------------------------------------------------
# CURRENT PNL %
# ---------------------------------------------------

final_df['Current_PnL_%'] = (
    final_df['Current_PnL']
    / final_df['Invested Amount']
) * 100

# ---------------------------------------------------
# ROUNDING
# ---------------------------------------------------

final_df['Approx_Buy_NAV'] = final_df['Approx_Buy_NAV'].round(2)

final_df['Current_PnL'] = final_df['Current_PnL'].round(2)

final_df['Current_PnL_%'] = final_df['Current_PnL_%'].round(2)

# ---------------------------------------------------
# LIST OF VALID MF COMPANIES
# ---------------------------------------------------

mf_companies = [
    "Axis",
    "HDFC",
    "SBI",
    "ICICI",
    "Kotak",
    "Nippon",
    "UTI",
    "DSP",
    "Groww",
    "Quant",
    "Mirae",
    "Franklin",
    "Canara",
    "Baroda BNP Paribas",
    "Tata",
    "Motilal",
    "Aditya Birla",
    "Edelweiss",
    "PGIM",
    "Bandhan",
    "Mahindra",
    "HSBC",
    "Parag Parikh",
    "JioBlackRock"
]

# ---------------------------------------------------
# FUNCTION TO EXTRACT MF COMPANY
# ---------------------------------------------------

# ---------------------------------------------------
# FUNCTION TO EXTRACT MF COMPANY
# CASE INSENSITIVE
# ---------------------------------------------------

def extract_mf_company(name):

    name = str(name).strip().lower()

    words = name.split()

    # Convert company list to lowercase once
    mf_companies_lower = [x.lower() for x in mf_companies]

    # Try 3-word match
    first_3 = " ".join(words[:3])

    # Try 2-word match
    first_2 = " ".join(words[:2])

    # Try 1-word match
    first_1 = words[0]

    if first_3 in mf_companies_lower:
        return mf_companies[
            mf_companies_lower.index(first_3)
        ]

    elif first_2 in mf_companies_lower:
        return mf_companies[
            mf_companies_lower.index(first_2)
        ]

    elif first_1 in mf_companies_lower:
        return mf_companies[
            mf_companies_lower.index(first_1)
        ]

    return "Other"

# ---------------------------------------------------
# CREATE MF_Company COLUMN
# ---------------------------------------------------

final_df['MF_Company'] = (
    final_df['Scheme_Name_Short']
    .apply(extract_mf_company)
)

# ---------------------------------------------------
# FUNCTION TO DETECT MF TYPE
# ---------------------------------------------------

def detect_mf_type(name):

    name = str(name).lower()

    if "small cap" in name or "smallcap" in name:
        return "Small Cap"
    
    elif "pharma" in name or "defense" in name:
        return "Sectoral"

    elif "large & mid" in name or "large and mid" in name:
        return "Large & Mid Cap"
    
    elif "mid cap" in name or "midcap" in name:
        return "Mid Cap"

    elif "large cap" in name:
        return "Large Cap"

    elif "multi" in name:
        return "Multi Cap"

    elif "flexi cap" in name:
        return "Flexi Cap"

    elif "contra" in name:
        return "Contra"

    elif "index" in name:
        return "Index"
    
    elif "value" in name:
        return "Value"

    elif "focused" in name:
        return "Focused"

    elif "etf" in name:
        return "ETF"

    elif "fof" in name:
        return "FOF"

    elif "hybrid" in name:
        return "Hybrid"

    elif "balanced" in name:
        return "Balanced"

    elif "debt" in name:
        return "Debt"
    
    elif "infra" in name:
        return "Infra"

    elif "liquid" in name:
        return "Liquid"

    elif "gold" in name:
        return "Gold"

    elif "sector" in name:
        return "Sectoral"

    elif "technology" in name:
        return "Technology"

    elif "consumption" in name:
        return "Consumption"

    elif "bank" in name:
        return "Banking"

    else:
        return "Other"

# ---------------------------------------------------
# CREATE MF_Type COLUMN
# ---------------------------------------------------

final_df['MF_Type'] = (
    final_df['Scheme_Name_Short']
    .apply(detect_mf_type)
)

# ---------------------------------------------------
# DISPLAY
# ---------------------------------------------------

print(final_df[[
    'Scheme Name',
    'Closing Bal (Units)',
    'Invested Amount',
    'Latest_NAV',
    'Approx_Buy_NAV',
    'Current_PnL',
    'Current_PnL_%',
    'MF_Company',
    'MF_Type'
]].head())

# ---------------------------------------------------
# SAVE UPDATED FILE
# ---------------------------------------------------

final_df.to_csv(
    "mutual_fund_latest_values.csv",
    index=False
)

print("\nUpdated file saved: mutual_fund_latest_values.csv")

# ---------------------------------------------------
# DISPLAY
# ---------------------------------------------------

print(final_df.head())


['Scheme Name', 'ISIN', 'Folio', 'Closing Bal (Units)', 'NAV', 'Invested Amount', 'Valuation', 'PnL', 'PnL %', 'Scheme_Name_Short', 'Key1', 'Key2', 'schemeCode', 'schemeName', 'isinGrowth', 'isinDivReinvestment', 'Priority']

Processing Scheme Code: 145110

Processing Scheme Code: 150266

Processing Scheme Code: 152056

Processing Scheme Code: 152056

Processing Scheme Code: 152770

Processing Scheme Code: 118989

Processing Scheme Code: 152082

Processing Scheme Code: 118955

Processing Scheme Code: 152082

Processing Scheme Code: 151130

Processing Scheme Code: 151036

Processing Scheme Code: 151113

Processing Scheme Code: 143903

Processing Scheme Code: 120591

Processing Scheme Code: 153859

Processing Scheme Code: 153787

Processing Scheme Code: 120160

Processing Scheme Code: 120160

Processing Scheme Code: 147704

Processing Scheme Code: 127042

Processing Scheme Code: 147704

Processing Scheme Code: 127042

Processing Scheme Code: 152712

Processing Scheme Code: 118632

Proces

In [42]:
import pandas as pd

# ---------------------------------------------------
# READ FILE
# ---------------------------------------------------

df = pd.read_csv("mutual_fund_latest_values.csv")

# ---------------------------------------------------
# ENSURE NUMERIC COLUMNS
# ---------------------------------------------------

numeric_cols = [
    'Invested Amount',
    'Current_Value',
    'Current_PnL'
]

for col in numeric_cols:

    df[col] = pd.to_numeric(
        df[col],
        errors='coerce'
    ).fillna(0)

# ---------------------------------------------------
# FUNCTION TO CREATE SUMMARY
# ---------------------------------------------------

def create_summary(group_col):

    summary = (
        df
        .groupby(group_col, dropna=False)
        .agg({
            'Invested Amount': 'sum',
            'Current_Value': 'sum',
            'Current_PnL': 'sum'
        })
        .reset_index()
    )

    # -----------------------------------------------
    # CALCULATE PnL %
    # -----------------------------------------------

    summary['Current_PnL_%'] = (
        summary['Current_PnL']
        / summary['Invested Amount']
    ) * 100

    # -----------------------------------------------
    # ROUNDING
    # -----------------------------------------------

    summary['Invested Amount'] = (
        summary['Invested Amount']
        .round(2)
    )

    summary['Current_Value'] = (
        summary['Current_Value']
        .round(2)
    )

    summary['Current_PnL'] = (
        summary['Current_PnL']
        .round(2)
    )

    summary['Current_PnL_%'] = (
        summary['Current_PnL_%']
        .round(2)
    )

    return summary

# ---------------------------------------------------
# CREATE SUMMARIES
# ---------------------------------------------------

summary_company = create_summary('MF_Company')

summary_type = create_summary('MF_Type')

summary_scheme = create_summary('Scheme_Name_Short')

# ---------------------------------------------------
# DISPLAY
# ---------------------------------------------------

print("\n==============================")
print("SUMMARY BY MF COMPANY")
print("==============================")

print(summary_company)

print("\n==============================")
print("SUMMARY BY MF TYPE")
print("==============================")

print(summary_type)

print("\n==============================")
print("SUMMARY BY SCHEME")
print("==============================")

print(summary_scheme)

# ---------------------------------------------------
# SAVE
# ---------------------------------------------------

summary_company.to_csv(
    "summary_by_company.csv",
    index=False
)

summary_type.to_csv(
    "summary_by_type.csv",
    index=False
)

summary_scheme.to_csv(
    "summary_by_scheme.csv",
    index=False
)

print("\nSummary files saved successfully.")


SUMMARY BY MF COMPANY
            MF_Company  Invested Amount  Current_Value  Current_PnL  \
0                 Axis           5000.0        6855.10      1855.10   
1   Baroda BNP Paribas           5000.0        5926.21       926.21   
2                  DSP         100000.0      116405.33     16405.33   
3                Groww          10000.0        9193.64      -806.36   
4                 HDFC         240000.0      280545.70     40545.70   
5                 HSBC         110000.0      137577.76     27577.76   
6                ICICI          55000.0       83465.61     28465.61   
7                Kotak         100000.0      132652.60     32652.60   
8              Motilal         660000.0      729335.83     69335.83   
9               Nippon         255000.0      309073.53     54073.53   
10               Other         440000.0      469149.95     29149.95   
11                PGIM          20000.0       26431.71      6431.71   
12               Quant         780000.0      800735.13